In [1]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), ".."))
main_dir = os.path.abspath('..')
os.chdir(main_dir)
sys.path.append(main_dir)

import numpy as np
import sympy as sp
import re, copy
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from PhysicsRegression import PhyReg

In [2]:
x_to_fit = []
y_to_fit = []

for Kp in ["0","1", "2", "3", "4", "5", "6"]:
    file_path = f"physical/data/data/physics_data/Convection_Efield_GSE_Kp={Kp}_50-quartile.txt"
    with open(file_path, "r") as fi:
        lines = [line.split() for line in fi.read().split("\n") if line.strip()] #no need to delete any line

    # 直接读取 GSE X, Y
    X_gsm = np.array([float(line[0]) for line in lines])
    Y_gsm = np.array([float(line[1]) for line in lines])
    Ey    = np.array([float(line[3]) for line in lines]).reshape(-1, 1)

    # === 关键修改：直接使用原始 GSE X, Y 作为输入特征！===
    x_raw = np.column_stack([X_gsm, Y_gsm])  # x_0 = X_gse, x_1 = Y_gse

    # 分割训练集（90%）
    x_train, _, y_train, _ = train_test_split(x_raw, Ey, test_size=0.1, random_state=42)
    
    # 可选：限制 3 ≤ r ≤ 6 Re 区域（推荐！）
    r = np.sqrt(x_train[:,0]**2 + x_train[:,1]**2)
    mask = (r >= 3.0) & (r <= 6.0)
    x_train = x_train[mask]
    y_train = y_train[mask]

    x_to_fit.append(x_train)
    y_to_fit.append(y_train)

print("数据加载完成！输入特征：x_0 = X_GSE, x_1 = Y_GSE")
print(f"每组训练样本数示例（Kp=4）：{len(x_to_fit[3])}")

数据加载完成！输入特征：x_0 = X_GSE, x_1 = Y_GSE
每组训练样本数示例（Kp=4）：604


In [3]:
phyreg = PhyReg(
    path="./model.pt",
    max_len=1000,
    device="cuda"
)

phyreg.fit(
    x_to_fit, y_to_fit,
    use_Divide=True,
    use_MCTS=True,
    use_GP=True,
    use_pysr_init=False,
    use_const_optimization=True,
    verbose=False,
    oracle_name="convection_gse_xy",
    oracle_file="./physical/data/oracle_gse_xy/",
    oracle_bs=12800,
    oracle_lr=0.005,
    oracle_epoch=1000,
    use_seperate_type=["id"],
    save_oracle_model=True
)

100%|██████████| 4000/4000 [00:29<00:00, 133.91it/s]


In [4]:
np.random.seed(2026)
num_bfgs = 700
current_refined_exprs = []
current_refined_mses = []
pbar = tqdm(total=7*7*num_bfgs, desc="Cross-dataset BFGS refining")

for i in range(7):
    current_refined_expr = []
    current_refined_mse = 0.0
    for j in range(7):
        best_expr = ""
        best_mse = 1e10
        expr = str(phyreg.best_gens[i]["predicted_tree"])
        expr = expr.replace("x_0", "(x_0 - 0.0)").replace("x_1", "(x_1 * 1.0)")
        for k,v in {"add": "+", "mul": "*", "sub": "-", "pow": "**", "inv": "1/", "neg": "-"}.items():
            expr = expr.replace(k, v)
        consts = re.findall(r"[+-]?\d+\.\d+(?:[eE][+-]?\d+)?", expr)
        for t,c in enumerate(consts):
            expr = expr.replace(c, f"c_{t}", 1)
        consts = np.array(list(map(float, consts)))

        x = x_to_fit[j]
        y = y_to_fit[j]

        for trial in range(num_bfgs):
            if len(consts) > 8: pbar.update(1); continue
            new_consts = consts + np.random.normal(0, 0.1, len(consts))
            new_expr = copy.deepcopy(expr)
            try:
                optimized = phyreg._const_optimize(new_expr, (x, y), len(consts), new_consts)
                optimized = np.clip(optimized, -1e4, 1e4)
            except:
                pbar.update(1)
                continue
            final_expr = new_expr
            for t,c in enumerate(optimized):
                final_expr = final_expr.replace(f"c_{t}", f"{c:.15f}", 1)
            try:
                f = sp.lambdify([sp.Symbol(f'x_{t}') for t in range(2)], final_expr, "numpy")
                pred = f(x[:,0], x[:,1])
                mse = np.mean((y.ravel() - pred.ravel())**2)
            except:
                mse = 1e10
            if mse < best_mse:
                best_mse = mse
                best_expr = final_expr
            pbar.update(1)

        current_refined_expr.append(best_expr)
        current_refined_mse += best_mse
    current_refined_exprs.append(current_refined_expr)
    current_refined_mses.append(current_refined_mse)

pbar.close()

idx = np.argmin(current_refined_mses)
refined_exprs = current_refined_exprs[idx]

print(f"\n the best: {idx+1} ，TOTLE MSE = {current_refined_mses[idx]:.4f}\n")
phyreg.express_skeleton(
    [{"predicted_tree": e} for e in refined_exprs],
    use_sp=True   
)

for k in range(7):
    expr = refined_exprs[k].replace("x_0", "x_0").replace("x_1", "x_1")
    print(f"Kp={k} |  → {expr[:500]}{'...' if len(expr)>200 else ''}")

Cross-dataset BFGS refining: 100%|██████████| 34300/34300 [04:00<00:00, 142.78it/s]  


 the best: 2 ，TOTLE MSE = 0.6169

idx          : 0
expr skeleton: -(C_0*cos(C_1*x_0 + C_2) + C_3)/(C_4*cos(C_5*x_1) - C_6)
constants    : 3.786 0.032 2.951 3.766 -2.488 0.601 3.219

idx          : 1
expr skeleton: -(C_0*cos(C_1*x_0 + C_2) + C_3)/(C_4*cos(C_5*x_1) - C_6)
constants    : 0.427 0.023 2.787 0.415 -0.313 0.639 0.451

idx          : 2
expr skeleton: -(C_0*cos(C_1*x_0 + C_2) + C_3)/(C_4*cos(C_5*x_1) - C_6)
constants    : 2.853 0.035 2.068 2.034 5.538 13.225 8.98

idx          : 3
expr skeleton: -(C_0*cos(C_1*x_0 + C_2) + C_3)/(C_4*cos(C_5*x_1) - C_6)
constants    : 0.14 0.498 0.994 0.369 -1.045 0.657 2.249

idx          : 4
expr skeleton: -(C_0*cos(C_1*x_0 + C_2) + C_3)/(C_4*cos(C_5*x_1) - C_6)
constants    : 0.357 0.651 0.397 0.748 0.178 1.582 2.02

idx          : 5
expr skeleton: -(C_0*cos(C_1*x_0 + C_2) + C_3)/(C_4*cos(C_5*x_1) - C_6)
constants    : 0.402 0.587 0.093 1.039 0.203 1.836 2.094

idx          : 6
expr skeleton: -(C_0*cos(C_1*x_0 + C_2) + C_3)/(C_4*cos(C_5*x_1) 